# Cadence Demo Notebook

This notebook provides an interactive overview of the Cadence evolutionary code synthesis pipeline.

Sections:
- Setup and Dependencies
- Directory Structure Overview
- Database Initialization and Inspection
- Task Interface and TSP Task
- LLM Interaction and Code Mutation
- Evolution Loop Execution
- Program Performance Evaluation
- Results Visualization
- Extending Cadence with a Custom Task
- Advanced Features: Async Evaluation and CLI

## 1. Setup and Dependencies

First, make sure you have a Python virtual environment and install the required dependencies.

```bash
# Create and activate virtual environment
python -m venv venv
source venv/bin/activate  # or venv\Scripts\activate on Windows

# Install project dependencies
uv pip install
```

Load environment variables and import core libraries:

In [ ]:
# Load environment variables and core imports
import os
from dotenv import load_dotenv

load_dotenv()

from src.database import Database, DatabaseConfig
from src.tasks.tsp_task import TSPTask
from src.evolver import apply_diff  # ensure evolve module is importable
from src.prompt_sampler import build
from src.llm import generate

## 2. Directory Structure Overview

Let's explore the `src/` folder and understand the key modules in Cadence:


In [ ]:
# Display project directory structure

for root, dirs, files in os.walk("src"):
    level = root.replace("src", "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

## 3. Baseline TSP Heuristics

Let's generate a small TSP instance and compare baseline heuristics: Nearest Neighbor vs. Reverse Tour.

In [ ]:
# Baseline Heuristics Comparison
from src.tasks.tsp_reference import nearest_neighbor, reversed_tour
from src.evaluator import generate_test_instance, compute_total_distance
import matplotlib.pyplot as plt

# Generate a small instance
cities = generate_test_instance(n=8, seed=0)

# Run heuristic solvers
nn_tour = nearest_neighbor(cities)
rev_tour = reversed_tour(cities)
nn_cost = compute_total_distance(nn_tour, cities)
rev_cost = compute_total_distance(rev_tour, cities)

# Display results
print(f"Nearest Neighbor cost: {nn_cost:.2f}")
print(f"Reversed Tour cost: {rev_cost:.2f}")

# Plot comparison
plt.figure(figsize=(6, 4))
plt.bar(["Nearest Neighbor", "Reversed"], [nn_cost, rev_cost], color=["blue", "orange"])
plt.ylabel("Tour Cost")
plt.title("Baseline Heuristic Comparison")
plt.show()

## 4. Database Initialization and Inspection

We can initialize the SQLite database, insert a sample program, and inspect stored entries.

In [ ]:
# Database Initialization and Inspection

# Initialize DB
db = Database(DatabaseConfig(database_path="demo_db.sqlite"))

# Insert a sample baseline program
sample_code = """### START_BLOCK\ndef tsp(cities): return list(range(len(cities)))\n### END_BLOCK"""
db.add_program(sample_code, metric=123.45, generation=0)

# Query stored programs
programs = db.get_all_programs()
for p in programs:
    print(p)

## 6. Evolution Loop Execution

Run a simple evolution loop over several generations and store results in the database.

In [ ]:
# Section 6: Evolution Loop Execution
from src.evaluator import Evaluator
from src.database import Database, DatabaseConfig

# Setup
task = TSPTask(n_cities=8)
evaluator = Evaluator()
db = Database(DatabaseConfig(database_path="demo_db.sqlite"))
parent_code = task.baseline_program

# Run evolution for 3 generations
for gen in range(1, 4):
    cities = task.generate_inputs(seed=gen)
    prompt = build(parent_code, cities)
    diffs = generate(prompt)
    if not diffs:
        break
    child_code = apply_diff(parent_code, [diffs[0]])
    res = evaluator.evaluate_code(child_code, task, seeds=[0])
    db.add_program(child_code, metric=res.cost, generation=gen)
    parent_code = child_code

print("Evolution complete. Programs stored in demo_db.sqlite.")

## 7. Program Performance Evaluation

Retrieve stored programs from the database and evaluate their performance metrics.

In [ ]:
# Section 7: Program Performance Evaluation
from src.database import Database, DatabaseConfig
from src.evaluator import Evaluator
from src.tasks.tsp_task import TSPTask

db = Database(DatabaseConfig(database_path="demo_db.sqlite"))
evaluator = Evaluator()
task = TSPTask(n_cities=8)

# Fetch and re-evaluate all programs
entries = db.get_all_programs()
for entry in entries:
    gen = entry.generation
    code = entry.code
    stored_cost = entry.metric
    cities = task.generate_inputs(seed=gen)
    eval_res = evaluator.evaluate_code(code, task, seeds=[0])
    print(
        f"Gen {gen} | Stored cost: {stored_cost:.2f} | Re-eval cost: {eval_res.cost:.2f} | Feasible: {eval_res.feasible}"
    )

## 8. Results Visualization

Visualize the evolution progress with a plot of cost over generations.

In [ ]:
# Section 8: Results Visualization
import matplotlib.pyplot as plt
from src.database import Database, DatabaseConfig

db = Database(DatabaseConfig(database_path="demo_db.sqlite"))
entries = db.get_all_programs()

# Extract data
gens = [e.generation for e in entries]
costs = [e.metric for e in entries]

plt.figure(figsize=(8, 4))
plt.plot(gens, costs, marker="o", linestyle="-")
plt.xlabel("Generation")
plt.ylabel("Tour Cost")
plt.title("Evolution Progress: Cost over Generations")
plt.grid(True)
plt.show()

## 9. Extending Cadence with a Custom Task

Show how to subclass `Task` to define a new optimization problem (e.g., sum-of-squares minimizer).

In [ ]:
# Section 9: Custom Task Example
from src.task import Task
from src.models import EvaluationResult


class SumSquaresTask(Task):
    @property
    def function_name(self) -> str:
        return "solve"

    def generate_inputs(self, seed: int):
        # Generate a list of numbers
        return [seed + i for i in range(5)]

    def evaluate(self, output, inputs) -> EvaluationResult:
        try:
            val = output(inputs)
            return EvaluationResult(cost=abs(val), feasible=True)
        except Exception as e:
            return EvaluationResult(cost=float("inf"), feasible=False, error=str(e))


# Test custom task
task = SumSquaresTask()


def solution(nums):
    return sum(x * x for x in nums)


res = task.evaluate(solution, task.generate_inputs(0))
print("Custom Task evaluation:", res)

## 10. Advanced Features: Async Evaluation and CLI

Demonstrate asynchronous evaluation and the command-line interface.

In [ ]:
# Section 10: Async Evaluation & CLI
import asyncio
from src.evaluator import Evaluator
from src.tasks.tsp_task import TSPTask


async def async_eval():
    evaluator = Evaluator()
    task = TSPTask(n_cities=6)
    seeds = [0, 1, 2, 3, 4]
    results = await asyncio.gather(
        *[
            asyncio.to_thread(evaluator.evaluate_code, task.baseline_program, task, [s])
            for s in seeds
        ]
    )
    print("Async eval results:", results)


asyncio.run(async_eval())

# CLI usage example:
# $ python main.py --task tsp --generations 5 --output